
# Exploratory Data Analysis of NIFTY 50 Futures and Options Market

## 1. Introduction

The Indian derivatives market plays an important role in financial markets by providing instruments for hedging, speculation, and risk management.

This project focuses on performing Exploratory Data Analysis (EDA) on historical NIFTY 50 Futures and Options market data obtained from the National Stock Exchange of India (NSE).

The analysis covers three categories of derivative instruments:

- NIFTY Index Futures
- NIFTY Call Options (CE)
- NIFTY Put Options (PE)

The study examines historical price movements, trading volume, open interest, strike prices, and expiry dates to identify patterns and relationships in derivatives market activity.

### 1.1 Study Period

January 2021 – September 2026

### 1.2 Data Source

National Stock Exchange of India (NSE)

https://www.nseindia.com/report-detail/fo_eq_security

## 2. Project Objectives

1. Understand the structure and characteristics of the datasets.
2. Analyze historical price movements of NIFTY futures and options.
3. Investigate trading volume and open interest patterns.
4. Examine the influence of strike prices and expiry dates on market activity.
5. Compare trading patterns between Call Options (CE) and Put Options (PE).
6. Identify statistical relationships between market variables.
7. Derive meaningful insights through data visualization and statistical analysis.

## 3. Project Methodology

The project follows these major stages:

1. Data collection and understanding.
2. Data cleaning and preprocessing.
3. Univariate and bivariate analysis.
4. Futures market analysis.
5. Call and Put options analysis.
6. Comparative and statistical analysis.
7. Interpretation of findings and conclusions.


In [2]:
# ============================================
# 1. IMPORTING REQUIRED LIBRARIES
# ============================================

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Statistical analysis
from scipy import stats
import statsmodels.api as sm

# File handling
import os
import zipfile
from pathlib import Path

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Visualization configuration
sns.set_theme(style="whitegrid", context="notebook")

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# ============================================
# 2. LOCATING RAW DATASETS
# ============================================

current_dir = Path.cwd()

print("Current working directory:", current_dir)

# Find the project root
if (current_dir / "data" / "raw").exists():
    PROJECT_ROOT = current_dir

elif (current_dir.parent / "data" / "raw").exists():
    PROJECT_ROOT = current_dir.parent

else:
    raise FileNotFoundError(
        "Could not locate data/raw directory. "
        "Check your project folder."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("\nProject root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)

# List available datasets
for file in sorted(RAW_DATA_DIR.glob("*.zip")):
    print(f"{file.name} — {file.stat().st_size / (1024**2):.2f} MB")

Current working directory: /home/yashraj/nifty50-FnO-eda/notebooks

Project root: /home/yashraj/nifty50-FnO-eda
Raw data directory: /home/yashraj/nifty50-FnO-eda/data/raw
Futures.zip — 0.17 MB
Options_CE.zip — 26.77 MB
Options_PE.zip — 27.53 MB


In [4]:
# ============================================
# 3. INSPECTING DATASET ARCHIVES
# ============================================

dataset_files = {
    "Futures": RAW_DATA_DIR / "Futures.zip",
    "Call Options (CE)": RAW_DATA_DIR / "Options_CE.zip",
    "Put Options (PE)": RAW_DATA_DIR / "Options_PE.zip"
}

for dataset_name, zip_path in dataset_files.items():

    print("\n" + "=" * 65)
    print(dataset_name)
    print("=" * 65)

    with zipfile.ZipFile(zip_path, 'r') as archive:

        csv_files = [
            name for name in archive.namelist()
            if name.lower().endswith('.csv')
        ]

        print("Total CSV files:", len(csv_files))

        print("\nSample files:")

        for filename in csv_files[:5]:
            print(" -", filename)


Futures
Total CSV files: 23

Sample files:
 - Futures/2021/FUTIDX_NIFTY_01-Jan-2021_TO_31-Mar-2021.csv
 - Futures/2021/FUTIDX_NIFTY_01-Apr-2021_TO_30-Jun-2021.csv
 - Futures/2021/FUTIDX_NIFTY_01-Jul-2021_TO_30-Sep-2021.csv
 - Futures/2021/FUTIDX_NIFTY_01-Oct-2021_TO_31-Dec-2021.csv
 - Futures/2022/FUTIDX_NIFTY_01-Apr-2022_TO_30-Jun-2022.csv

Call Options (CE)
Total CSV files: 23

Sample files:
 - Options_CE/2021/OPTIDX_NIFTY_CE_01-Oct-2021_TO_31-Dec-2021.csv
 - Options_CE/2021/OPTIDX_NIFTY_CE_01-Jan-2021_TO_31-Mar-2021.csv
 - Options_CE/2021/OPTIDX_NIFTY_CE_01-Apr-2021_TO_30-Jun-2021.csv
 - Options_CE/2021/OPTIDX_NIFTY_CE_01-Jul-2021_TO_30-Sep-2021.csv
 - Options_CE/2022/OPTIDX_NIFTY_CE_01-Jan-2022_TO_31-Mar-2022.csv

Put Options (PE)
Total CSV files: 23

Sample files:
 - Options_PE/2021/OPTIDX_NIFTY_PE_01-Jul-2021_TO_30-Sep-2021.csv
 - Options_PE/2021/OPTIDX_NIFTY_PE_01-Jan-2021_TO_31-Mar-2021.csv
 - Options_PE/2021/OPTIDX_NIFTY_PE_01-Apr-2021_TO_30-Jun-2021.csv
 - Options_PE/2021/OP

In [5]:
# ============================================
# 4. DATASET SUMMARY
# ============================================

dataset_summary = []

for dataset_name, zip_path in dataset_files.items():

    with zipfile.ZipFile(zip_path, 'r') as archive:

        csv_files = [
            name for name in archive.namelist()
            if name.lower().endswith('.csv')
        ]

        dataset_summary.append({
            "Dataset": dataset_name,
            "Number of CSV Files": len(csv_files),
            "Compressed Size (MB)": round(
                zip_path.stat().st_size / (1024**2), 2
            )
        })

summary_df = pd.DataFrame(dataset_summary)

display(summary_df)

,Dataset,Number of CSV Files,Compressed Size (MB)
0,Futures,23,0.17
1,Call Options (CE),23,26.77
2,Put Options (PE),23,27.53



## 5. Dataset Structure and Schema Analysis

This section examines the internal structure of the three datasets. The objective is to understand the available variables, their data types, and the consistency of the quarterly files before beginning data preprocessing.


In [6]:
# ============================================
# 5.1 READING SAMPLE CSV FILES
# ============================================

sample_data = {}

for dataset_name, zip_path in dataset_files.items():

    with zipfile.ZipFile(zip_path, 'r') as archive:

        csv_files = sorted([
            name for name in archive.namelist()
            if name.lower().endswith('.csv')
        ])

        sample_file = csv_files[0]

        with archive.open(sample_file) as file:
            sample_df = pd.read_csv(file)

        sample_data[dataset_name] = sample_df

        print("\n" + "=" * 70)
        print(dataset_name)
        print("=" * 70)

        print("Sample file:", sample_file)
        print("Shape:", sample_df.shape)

        display(sample_df.head())


Futures
Sample file: Futures/2021/FUTIDX_NIFTY_01-Apr-2021_TO_30-Jun-2021.csv
Shape: (186, 14)


,Symbol,Date,Expiry,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * in ₹ Lakhs,Open Int,Change in OI,Underlying Value
0,NIFTY,30-Jun-2021,29-Jul-2021,15816.05,15876.20,15737.05,15752.15,15747.00,15752.15,134926.00,1066811.48,9657400.00,227650.00,-
1,NIFTY,30-Jun-2021,30-Sep-2021,15900.60,15968.35,15839.00,15853.30,15851.80,15853.30,888.00,7064.66,45900.00,10300.00,-
2,NIFTY,30-Jun-2021,26-Aug-2021,15843.40,15918.00,15787.30,15800.80,15798.10,15800.80,3782.00,29987.09,455500.00,23750.00,-
3,NIFTY,29-Jun-2021,29-Jul-2021,15843.10,15860.00,15758.70,15785.60,15795.00,15785.60,118797.00,938402.39,9429750.00,-46650.00,15748.45
4,NIFTY,29-Jun-2021,26-Aug-2021,15866.40,15907.70,15809.00,15832.25,15840.25,15832.25,3999.00,31680.66,431750.00,5100.00,15748.45



Call Options (CE)
Sample file: Options_CE/2021/OPTIDX_NIFTY_CE_01-Apr-2021_TO_30-Jun-2021.csv
Shape: (59579, 17)


,Symbol,Date,Expiry,Option type,Strike Price,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * in ₹ Lakhs,Premium Turnover ** in ₹ Lakhs,Open Int,Change in OI,Underlying Value
0,NIFTY,30-Jun-2021,01-Jul-2021,CE,"16,250.00",2.10,2.10,0.80,1.20,1.00,1.20,26524.00,323289.28,28.03,600600.00,-79800.00,-
1,NIFTY,30-Jun-2021,01-Jul-2021,CE,"14,900.00",897.70,935.45,897.70,935.45,935.45,823.00,25.00,296.63,17.25,900.00,-675.00,-
2,NIFTY,30-Jun-2021,01-Jul-2021,CE,"15,600.00",187.50,243.95,122.00,133.15,131.00,133.15,50609.00,599134.84,7009.54,268350.00,-43800.00,-
3,NIFTY,30-Jun-2021,01-Jul-2021,CE,"16,300.00",1.15,1.95,0.65,1.00,0.70,1.00,127136.00,1554354.74,117.14,1974300.00,-1246125.00,-
4,NIFTY,30-Jun-2021,01-Jul-2021,CE,"14,950.00",-,-,-,890.35,890.35,773.00,-,-,-,150.00,-,-



Put Options (PE)
Sample file: Options_PE/2021/OPTIDX_NIFTY_PE_01-Apr-2021_TO_30-Jun-2021.csv
Shape: (64555, 17)


,Symbol,Date,Expiry,Option type,Strike Price,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * in ₹ Lakhs,Premium Turnover ** in ₹ Lakhs,Open Int,Change in OI,Underlying Value
0,NIFTY,30-Jun-2021,01-Jul-2021,PE,"14,050.00",0.15,0.30,0.10,0.10,0.10,0.10,15.00,158.06,-,2250.00,-75.00,-
1,NIFTY,30-Jun-2021,01-Jul-2021,PE,"13,350.00",0.10,0.10,0.10,0.10,0.10,-,1.00,10.01,-,600.00,-,-
2,NIFTY,30-Jun-2021,01-Jul-2021,PE,"14,100.00",0.45,0.45,0.15,0.20,0.15,0.20,964.00,10194.49,0.19,10125.00,-5175.00,-
3,NIFTY,30-Jun-2021,01-Jul-2021,PE,"13,400.00",0.10,0.30,0.05,0.15,0.05,0.15,1848.00,18572.69,0.29,19425.00,-9675.00,-
4,NIFTY,30-Jun-2021,01-Jul-2021,PE,"14,150.00",0.15,0.50,0.15,0.20,0.20,0.20,1613.00,17118.32,0.36,375.00,300.00,-


In [9]:
# ============================================
# 5.2 COLUMN NAME ANALYSIS
# ============================================

for dataset_name, df in sample_data.items():

    print("\n", "=" * 60)
    print(dataset_name)
    print("=" * 60)

    for i, column in enumerate(df.columns, start=1):
        print(f"{i}. {column}")


Futures
1. Symbol  
2. Date  
3. Expiry  
4. Open  
5. High  
6. Low  
7. Close  
8. LTP  
9. Settle Price  
10. No. of contracts  
11. Turnover * in   ₹ Lakhs
12. Open Int  
13. Change in OI  
14. Underlying Value  

Call Options (CE)
1. Symbol  
2. Date  
3. Expiry  
4. Option type  
5. Strike Price  
6. Open  
7. High  
8. Low  
9. Close  
10. LTP  
11. Settle Price  
12. No. of contracts  
13. Turnover * in  ₹ Lakhs
14. Premium Turnover ** in   ₹ Lakhs
15. Open Int  
16. Change in OI  
17. Underlying Value  

Put Options (PE)
1. Symbol  
2. Date  
3. Expiry  
4. Option type  
5. Strike Price  
6. Open  
7. High  
8. Low  
9. Close  
10. LTP  
11. Settle Price  
12. No. of contracts  
13. Turnover * in  ₹ Lakhs
14. Premium Turnover ** in   ₹ Lakhs
15. Open Int  
16. Change in OI  
17. Underlying Value  


In [10]:
# ============================================
# 5.3 DATA TYPE ANALYSIS
# ============================================

for dataset_name, df in sample_data.items():

    print("\n", "=" * 60)
    print(dataset_name)
    print("=" * 60)

    dtype_table = pd.DataFrame({
        'Column': df.columns,
        'Data Type': df.dtypes.astype(str).values,
        'Non-Null Count': df.notna().sum().values,
        'Missing Count': df.isna().sum().values
    })

    display(dtype_table)


Futures


,Column,Data Type,Non-Null Count,Missing Count
0,Symbol,str,186,0
1,Date,str,186,0
2,Expiry,str,186,0
3,Open,str,186,0
4,High,str,186,0
5,Low,str,186,0
6,Close,str,186,0
7,LTP,str,186,0
8,Settle Price,str,186,0
9,No. of contracts,str,186,0



Call Options (CE)


,Column,Data Type,Non-Null Count,Missing Count
0,Symbol,str,59579,0
1,Date,str,59579,0
2,Expiry,str,59579,0
3,Option type,str,59579,0
4,Strike Price,float64,59579,0
5,Open,str,59579,0
6,High,str,59579,0
7,Low,str,59579,0
8,Close,str,59579,0
9,LTP,str,59579,0



Put Options (PE)


,Column,Data Type,Non-Null Count,Missing Count
0,Symbol,str,64555,0
1,Date,str,64555,0
2,Expiry,str,64555,0
3,Option type,str,64555,0
4,Strike Price,float64,64555,0
5,Open,str,64555,0
6,High,str,64555,0
7,Low,str,64555,0
8,Close,str,64555,0
9,LTP,str,64555,0


In [11]:
# ============================================
# 5.4 SCHEMA CONSISTENCY CHECK
# ============================================

schema_results = []

for dataset_name, zip_path in dataset_files.items():

    with zipfile.ZipFile(zip_path, 'r') as archive:

        csv_files = sorted([
            name for name in archive.namelist()
            if name.lower().endswith('.csv')
        ])

        reference_columns = None

        for filename in csv_files:

            with archive.open(filename) as file:
                df_temp = pd.read_csv(file, nrows=0)

            columns = list(df_temp.columns)

            if reference_columns is None:
                reference_columns = columns

            schema_results.append({
                'Dataset': dataset_name,
                'File': filename,
                'Column Count': len(columns),
                'Matches Reference Schema':
                    set(columns) == set(reference_columns),
                'Missing Columns':
                    list(set(reference_columns) - set(columns)),
                'Additional Columns':
                    list(set(columns) - set(reference_columns))
            })

schema_df = pd.DataFrame(schema_results)

display(schema_df)

,Dataset,File,Column Count,Matches Reference Schema,Missing Columns,Additional Columns
0,Futures,Futures/2021/FUTIDX_NIFTY_01-Apr-2021_TO_30-Ju...,14,True,[],[]
1,Futures,Futures/2021/FUTIDX_NIFTY_01-Jan-2021_TO_31-Ma...,14,True,[],[]
2,Futures,Futures/2021/FUTIDX_NIFTY_01-Jul-2021_TO_30-Se...,14,True,[],[]
3,Futures,Futures/2021/FUTIDX_NIFTY_01-Oct-2021_TO_31-De...,14,True,[],[]
4,Futures,Futures/2022/FUTIDX_NIFTY_01-Apr-2022_TO_30-Ju...,14,True,[],[]
5,Futures,Futures/2022/FUTIDX_NIFTY_01-Jan-2022_TO_31-Ma...,14,True,[],[]
6,Futures,Futures/2022/FUTIDX_NIFTY_01-Jul-2022_TO_30-Se...,14,True,[],[]
7,Futures,Futures/2022/FUTIDX_NIFTY_01-Oct-2022_TO_31-De...,14,True,[],[]
8,Futures,Futures/2023/FUTIDX_NIFTY_01-Apr-2023_TO_30-Ju...,14,True,[],[]
9,Futures,Futures/2023/FUTIDX_NIFTY_01-Jan-2023_TO_31-Ma...,14,True,[],[]


In [12]:
schema_df[
    schema_df['Matches Reference Schema'] == False
]

,Dataset,File,Column Count,Matches Reference Schema,Missing Columns,Additional Columns


Therefore, all files within each dataset have matching column names.

In [13]:
# ============================================
# 5.5 INITIAL DESCRIPTIVE STATISTICS
# ============================================

for dataset_name, df in sample_data.items():

    print("\n", "=" * 60)
    print(dataset_name)
    print("=" * 60)

    display(df.describe(include='all').T)


Futures


,count,unique,top,freq
Symbol,186,1,NIFTY,186
Date,186,61,24-Jun-2021,4
Expiry,186,6,24-Jun-2021,57
Open,186,180,-,3
High,186,179,-,3
Low,186,180,-,3
Close,186,183,-,3
LTP,186,179,-,3
Settle Price,186,183,-,3
No. of contracts,186,184,-,3



Call Options (CE)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Symbol,59579,1,NIFTY,59579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,59579,61,17-Jun-2021,1085,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Expiry,59579,34,24-Jun-2021,5960,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Option type,59579,1,CE,59579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Strike Price,"59,579.00",NaN,NaN,NaN,"14,743.16","1,574.21","8,500.00","13,700.00","14,850.00","15,950.00","18,500.00"
Open,59579,8589,-,43515,NaN,NaN,NaN,NaN,NaN,NaN,NaN
High,59579,8924,-,43515,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Low,59579,8295,-,43515,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Close,59579,10746,-,375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LTP,59579,8460,-,38272,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Put Options (PE)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Symbol,64555,1,NIFTY,64555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,64555,61,17-Jun-2021,1167,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Expiry,64555,34,24-Jun-2021,6815,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Option type,64555,1,PE,64555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Strike Price,"64,555.00",NaN,NaN,NaN,"14,429.13","1,915.11","6,000.00","13,400.00","14,650.00","15,850.00","18,500.00"
Open,64555,7870,-,46040,NaN,NaN,NaN,NaN,NaN,NaN,NaN
High,64555,8168,-,46040,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Low,64555,7464,-,46040,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Close,64555,9844,-,375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LTP,64555,7621,-,35541,NaN,NaN,NaN,NaN,NaN,NaN,NaN



## 6. Full Dataset Loading

After examining the sample records and validating the column structures, all quarterly CSV files are combined into three consolidated datasets.

The original values are preserved during this stage to avoid unintended data modifications. A source-file identifier is also retained for traceability and validation.


In [14]:
# ============================================
# 6.1 LOADING ALL QUARTERLY CSV FILES
# ============================================

def load_full_dataset(zip_path, dataset_name):

    dataframes = []

    with zipfile.ZipFile(zip_path, 'r') as archive:

        csv_files = sorted([
            name for name in archive.namelist()
            if name.lower().endswith('.csv')
        ])

        for filename in csv_files:

            with archive.open(filename) as file:

                df = pd.read_csv(
                    file,
                    dtype=str,
                    keep_default_na=False,
                    low_memory=False
                )

            # Standardize whitespace in column names
            df.columns = df.columns.str.strip()

            # Retain source information
            df['source_file'] = filename

            dataframes.append(df)

    combined_df = pd.concat(
        dataframes,
        ignore_index=True
    )

    print(f"{dataset_name} loaded successfully")
    print(f"Files processed: {len(csv_files):,}")
    print(f"Total records: {len(combined_df):,}")
    print(f"Total columns: {len(combined_df.columns):,}")

    return combined_df


# Load all three datasets

futures_df = load_full_dataset(
    dataset_files["Futures"],
    "Futures"
)

calls_df = load_full_dataset(
    dataset_files["Call Options (CE)"],
    "Call Options (CE)"
)

puts_df = load_full_dataset(
    dataset_files["Put Options (PE)"],
    "Put Options (PE)"
)

Futures loaded successfully
Files processed: 23
Total records: 4,348
Total columns: 15
Call Options (CE) loaded successfully
Files processed: 23
Total records: 1,319,036
Total columns: 18
Put Options (PE) loaded successfully
Files processed: 23
Total records: 1,358,362
Total columns: 18


In [15]:
# ============================================
# 6.2 DATASET DIMENSION VALIDATION
# ============================================

full_dataset_summary = pd.DataFrame([
    {
        "Dataset": "Futures",
        "Rows": futures_df.shape[0],
        "Columns": futures_df.shape[1] - 1
    },
    {
        "Dataset": "Call Options (CE)",
        "Rows": calls_df.shape[0],
        "Columns": calls_df.shape[1] - 1
    },
    {
        "Dataset": "Put Options (PE)",
        "Rows": puts_df.shape[0],
        "Columns": puts_df.shape[1] - 1
    }
])

display(full_dataset_summary)

print(
    "Total records across all datasets:",
    f"{len(futures_df) + len(calls_df) + len(puts_df):,}"
)

,Dataset,Rows,Columns
0,Futures,4348,14
1,Call Options (CE),1319036,17
2,Put Options (PE),1358362,17


Total records across all datasets: 2,681,746


In [16]:
# ============================================
# 6.3 RECORD COUNT BY SOURCE FILE
# ============================================

def file_record_summary(df, dataset_name):

    summary = (
        df.groupby('source_file')
        .size()
        .reset_index(name='Record Count')
        .sort_values('source_file')
    )

    print(f"\n{dataset_name} — Records per CSV file")
    display(summary)

    return summary


futures_file_summary = file_record_summary(
    futures_df, "Futures"
)

calls_file_summary = file_record_summary(
    calls_df, "Call Options"
)

puts_file_summary = file_record_summary(
    puts_df, "Put Options"
)


Futures — Records per CSV file


,source_file,Record Count
0,Futures/2021/FUTIDX_NIFTY_01-Apr-2021_TO_30-Ju...,186
1,Futures/2021/FUTIDX_NIFTY_01-Jan-2021_TO_31-Ma...,186
2,Futures/2021/FUTIDX_NIFTY_01-Jul-2021_TO_30-Se...,191
3,Futures/2021/FUTIDX_NIFTY_01-Oct-2021_TO_31-De...,190
4,Futures/2022/FUTIDX_NIFTY_01-Apr-2022_TO_30-Ju...,186
5,Futures/2022/FUTIDX_NIFTY_01-Jan-2022_TO_31-Ma...,184
6,Futures/2022/FUTIDX_NIFTY_01-Jul-2022_TO_30-Se...,191
7,Futures/2022/FUTIDX_NIFTY_01-Oct-2022_TO_31-De...,185
8,Futures/2023/FUTIDX_NIFTY_01-Apr-2023_TO_30-Ju...,183
9,Futures/2023/FUTIDX_NIFTY_01-Jan-2023_TO_31-Ma...,186



Call Options — Records per CSV file


,source_file,Record Count
0,Options_CE/2021/OPTIDX_NIFTY_CE_01-Apr-2021_TO...,59579
1,Options_CE/2021/OPTIDX_NIFTY_CE_01-Jan-2021_TO...,64283
2,Options_CE/2021/OPTIDX_NIFTY_CE_01-Jul-2021_TO...,59556
3,Options_CE/2021/OPTIDX_NIFTY_CE_01-Oct-2021_TO...,64919
4,Options_CE/2022/OPTIDX_NIFTY_CE_01-Apr-2022_TO...,66573
5,Options_CE/2022/OPTIDX_NIFTY_CE_01-Jan-2022_TO...,64310
6,Options_CE/2022/OPTIDX_NIFTY_CE_01-Jul-2022_TO...,75602
7,Options_CE/2022/OPTIDX_NIFTY_CE_01-Oct-2022_TO...,71048
8,Options_CE/2023/OPTIDX_NIFTY_CE_01-Apr-2023_TO...,41544
9,Options_CE/2023/OPTIDX_NIFTY_CE_01-Jan-2023_TO...,43514



Put Options — Records per CSV file


,source_file,Record Count
0,Options_PE/2021/OPTIDX_NIFTY_PE_01-Apr-2021_TO...,64555
1,Options_PE/2021/OPTIDX_NIFTY_PE_01-Jan-2021_TO...,69305
2,Options_PE/2021/OPTIDX_NIFTY_PE_01-Jul-2021_TO...,64218
3,Options_PE/2021/OPTIDX_NIFTY_PE_01-Oct-2021_TO...,69492
4,Options_PE/2022/OPTIDX_NIFTY_PE_01-Apr-2022_TO...,69219
5,Options_PE/2022/OPTIDX_NIFTY_PE_01-Jan-2022_TO...,67131
6,Options_PE/2022/OPTIDX_NIFTY_PE_01-Jul-2022_TO...,77072
7,Options_PE/2022/OPTIDX_NIFTY_PE_01-Oct-2022_TO...,72531
8,Options_PE/2023/OPTIDX_NIFTY_PE_01-Apr-2023_TO...,41513
9,Options_PE/2023/OPTIDX_NIFTY_PE_01-Jan-2023_TO...,43569


In [17]:
# ============================================
# 6.4 MEMORY USAGE ANALYSIS
# ============================================

memory_summary = pd.DataFrame({
    "Dataset": [
        "Futures",
        "Call Options",
        "Put Options"
    ],
    "Memory Usage (MB)": [
        futures_df.memory_usage(deep=True).sum() / (1024**2),
        calls_df.memory_usage(deep=True).sum() / (1024**2),
        puts_df.memory_usage(deep=True).sum() / (1024**2)
    ]
})

memory_summary["Memory Usage (MB)"] = (
    memory_summary["Memory Usage (MB)"].round(2)
)

display(memory_summary)

,Dataset,Memory Usage (MB)
0,Futures,1.21
1,Call Options,365.88
2,Put Options,376.46


## 7. Data Quality Assessment

Before performing data cleaning and statistical analysis, it is important to assess the quality of the complete datasets.

This section investigates missing values, placeholder entries, duplicate records, date validity, and inconsistencies in numerical fields. The objective is to identify potential data quality issues while preserving the original observations and avoiding assumptions about the meaning of missing or placeholder values.

The findings from this assessment will guide the subsequent data preprocessing stage.

In [18]:
# ============================================
# 7.1 MISSING AND PLACEHOLDER VALUE ANALYSIS
# ============================================

def missing_value_analysis(df, dataset_name):

    results = []

    for column in df.columns:

        missing_count = df[column].isna().sum()

        placeholder_count = (
            df[column]
            .astype(str)
            .str.strip()
            .eq('-')
            .sum()
        )

        blank_count = (
            df[column]
            .astype(str)
            .str.strip()
            .eq('')
            .sum()
        )

        total_issues = (
            missing_count
            + placeholder_count
            + blank_count
        )

        results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Actual Missing": missing_count,
            "Hyphen Placeholders": placeholder_count,
            "Blank Values": blank_count,
            "Total Potential Missing": total_issues,
            "Missing Percentage": (
                total_issues / len(df) * 100
            )
        })

    return pd.DataFrame(results)


missing_summary = pd.concat(
    [
        missing_value_analysis(futures_df, "Futures"),
        missing_value_analysis(calls_df, "Call Options (CE)"),
        missing_value_analysis(puts_df, "Put Options (PE)")
    ],
    ignore_index=True
)

display(
    missing_summary.sort_values(
        by=["Dataset", "Missing Percentage"],
        ascending=[True, False]
    )
)

,Dataset,Column,Actual Missing,Hyphen Placeholders,Blank Values,Total Potential Missing,Missing Percentage
30,Call Options (CE),Change in OI,0,769337,0,769337,58.33
28,Call Options (CE),Premium Turnover ** in ₹ Lakhs,0,741861,0,741861,56.24
26,Call Options (CE),No. of contracts,0,740890,0,740890,56.17
20,Call Options (CE),Open,0,740793,0,740793,56.16
21,Call Options (CE),High,0,740793,0,740793,56.16
22,Call Options (CE),Low,0,740793,0,740793,56.16
27,Call Options (CE),Turnover * in ₹ Lakhs,0,740793,0,740793,56.16
29,Call Options (CE),Open Int,0,607636,0,607636,46.07
24,Call Options (CE),LTP,0,599792,0,599792,45.47
31,Call Options (CE),Underlying Value,0,506233,0,506233,38.38


### 7.2 Duplicate Record Analysis

Duplicate records can affect statistical summaries and lead to misleading conclusions. This section checks for both exact duplicate rows and repeated contract identifiers.

Exact duplicates are identified using all original dataset columns, excluding the source-file metadata. Contract-level duplicates are identified using the relevant instrument identifiers and trading date.

In [20]:
# ============================================
# 7.2 DUPLICATE RECORD ANALYSIS
# ============================================

def duplicate_analysis(df, dataset_name, key_columns):

    original_columns = [
        col for col in df.columns
        if col != "source_file"
    ]

    # Exact duplicate rows
    exact_duplicate_mask = df.duplicated(
        subset=original_columns,
        keep="first"
    )

    exact_duplicates = df.loc[exact_duplicate_mask].copy()

    # Contract-level duplicate identifiers
    normalized_keys = df[key_columns].astype(str).apply(
        lambda col: col.str.strip()
    )

    repeated_key_mask = normalized_keys.duplicated(
        keep=False
    )

    repeated_keys = df.loc[repeated_key_mask].copy()

    # Market-data columns
    market_columns = [
        col for col in original_columns
        if col not in key_columns
    ]

    # Rows where all market fields contain '-'
    all_hyphen_mask = (
        df[market_columns]
        .astype(str)
        .apply(lambda col: col.str.strip().eq('-'))
        .all(axis=1)
    )

    print(f"\n{dataset_name}")
    print("=" * 60)
    print(f"Total records: {len(df):,}")
    print(f"Exact duplicate rows: {exact_duplicate_mask.sum():,}")
    print(f"Rows in repeated contract-key groups: {repeated_key_mask.sum():,}")
    print(f"Repeated contract-key groups: {normalized_keys[repeated_key_mask].drop_duplicates().shape[0]:,}")
    print(f"Rows with all market fields as '-': {all_hyphen_mask.sum():,}")

    # Inspect repeated keys with differing original values
    differing_groups = (
        repeated_keys
        .groupby(key_columns, dropna=False)[original_columns]
        .apply(lambda group: len(group.drop_duplicates()) > 1)
    )

    print(
        "Repeated contract-key groups with differing records:",
        int(differing_groups.sum())
    )

    return {
        "exact_duplicates": exact_duplicates,
        "repeated_keys": repeated_keys,
        "all_hyphen_mask": all_hyphen_mask,
        "differing_groups": differing_groups
    }


futures_duplicate_results = duplicate_analysis(
    futures_df,
    "Futures",
    ["Symbol", "Date", "Expiry"]
)

calls_duplicate_results = duplicate_analysis(
    calls_df,
    "Call Options (CE)",
    ["Symbol", "Date", "Expiry", "Option type", "Strike Price"]
)

puts_duplicate_results = duplicate_analysis(
    puts_df,
    "Put Options (PE)",
    ["Symbol", "Date", "Expiry", "Option type", "Strike Price"]
)


Futures
Total records: 4,348
Exact duplicate rows: 54
Rows in repeated contract-key groups: 58
Repeated contract-key groups: 4
Rows with all market fields as '-': 83
Repeated contract-key groups with differing records: 0

Call Options (CE)
Total records: 1,319,036
Exact duplicate rows: 57,226
Rows in repeated contract-key groups: 60,138
Repeated contract-key groups: 2,907
Rows with all market fields as '-': 68,221
Repeated contract-key groups with differing records: 5

Put Options (PE)
Total records: 1,358,362
Exact duplicate rows: 57,673
Rows in repeated contract-key groups: 60,583
Repeated contract-key groups: 2,905
Rows with all market fields as '-': 68,248
Repeated contract-key groups with differing records: 5


In [21]:
# Inspect non-identical repeated contract records

for name, results, df, key_columns in [
    ("CE", calls_duplicate_results, calls_df,
     ["Symbol", "Date", "Expiry", "Option type", "Strike Price"]),

    ("PE", puts_duplicate_results, puts_df,
     ["Symbol", "Date", "Expiry", "Option type", "Strike Price"])
]:

    print(f"\n{name}: Non-identical repeated contract records")

    differing_keys = results["differing_groups"]
    differing_keys = differing_keys[differing_keys].index

    if len(differing_keys) > 0:
        sample_key = differing_keys[0]
        sample_key = sample_key if isinstance(sample_key, tuple) else (sample_key,)

        mask = pd.Series(True, index=df.index)

        for column, value in zip(key_columns, sample_key):
            mask &= df[column].astype(str).str.strip().eq(str(value).strip())

        display(df.loc[mask].head(20))


CE: Non-identical repeated contract records


,Symbol,Date,Expiry,Option type,Strike Price,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * in ₹ Lakhs,Premium Turnover ** in ₹ Lakhs,Open Int,Change in OI,Underlying Value,source_file
956622,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956627,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,-,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956632,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956637,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956662,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956667,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956672,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956677,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956770,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...
956771,NIFTY,23-May-2025,03-Jul-2025,CE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_CE/2025/OPTIDX_NIFTY_CE_01-Apr-2025_TO...



PE: Non-identical repeated contract records


,Symbol,Date,Expiry,Option type,Strike Price,Open,High,Low,Close,LTP,Settle Price,No. of contracts,Turnover * in ₹ Lakhs,Premium Turnover ** in ₹ Lakhs,Open Int,Change in OI,Underlying Value,source_file
988422,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988427,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,-,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988432,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988437,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988442,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988472,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988477,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988579,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988580,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...
988585,NIFTY,23-May-2025,03-Jul-2025,PE,29900.00,-,-,-,-,-,-,-,-,-,-,-,24853.15,Options_PE/2025/OPTIDX_NIFTY_PE_01-Apr-2025_TO...


### 7.3 Date Validation and Study Period

This section examines the trading-date and expiry-date fields across all three datasets.

The analysis checks whether dates can be parsed successfully, identifies invalid or missing dates, determines the observed date range, and verifies whether the records fall within the intended study period of January 2021 to September 2026.

The original datasets remain unchanged. Parsed dates are created separately for validation.

In [22]:
# ============================================
# 7.3 DATE VALIDATION AND STUDY PERIOD
# ============================================

def find_date_column(df, candidates):
    """Find a date column without assuming exact capitalization."""

    normalized = {
        col.strip().lower(): col
        for col in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    return None


def date_validation(df, dataset_name):

    trading_col = find_date_column(
        df, ["Date", "Trading Date", "Trade Date"]
    )

    expiry_col = find_date_column(
        df, ["Expiry", "Expiry Date", "Expiration Date"]
    )

    print(f"\n{dataset_name}")
    print("=" * 65)

    results = {}

    for date_type, column in [
        ("Trading Date", trading_col),
        ("Expiry Date", expiry_col)
    ]:

        if column is None:
            print(f"{date_type}: Column not found.")
            continue

        parsed_dates = pd.to_datetime(
            df[column].astype(str).str.strip(),
            errors="coerce",
            dayfirst=True
        )

        invalid_count = parsed_dates.isna().sum()
        valid_dates = parsed_dates.dropna()

        results[date_type] = parsed_dates

        print(f"\n{date_type} — Original column: {column}")
        print(f"Valid dates: {len(valid_dates):,}")
        print(f"Invalid or missing dates: {invalid_count:,}")

        if not valid_dates.empty:
            print(f"Earliest date: {valid_dates.min()}")
            print(f"Latest date:   {valid_dates.max()}")

            outside_period = (
                (valid_dates < "2021-01-01")
                | (valid_dates > "2026-09-30")
            )

            print(
                "Valid dates outside intended study period:",
                f"{outside_period.sum():,}"
            )

    return results


futures_date_results = date_validation(
    futures_df, "Futures"
)

calls_date_results = date_validation(
    calls_df, "Call Options (CE)"
)

puts_date_results = date_validation(
    puts_df, "Put Options (PE)"
)


Futures

Trading Date — Original column: Date
Valid dates: 4,348
Invalid or missing dates: 0
Earliest date: 2021-01-01 00:00:00
Latest date:   2026-09-25 00:00:00
Valid dates outside intended study period: 0

Expiry Date — Original column: Expiry
Valid dates: 4,348
Invalid or missing dates: 0
Earliest date: 2021-01-28 00:00:00
Latest date:   2026-11-23 00:00:00
Valid dates outside intended study period: 64

Call Options (CE)

Trading Date — Original column: Date
Valid dates: 1,319,036
Invalid or missing dates: 0
Earliest date: 2021-01-01 00:00:00
Latest date:   2026-09-25 00:00:00
Valid dates outside intended study period: 0

Expiry Date — Original column: Expiry
Valid dates: 1,319,036
Invalid or missing dates: 0
Earliest date: 2021-01-07 00:00:00
Latest date:   2031-06-24 00:00:00
Valid dates outside intended study period: 95,203

Put Options (PE)

Trading Date — Original column: Date
Valid dates: 1,358,362
Invalid or missing dates: 0
Earliest date: 2021-01-01 00:00:00
Latest date:  

### 7.4 Numeric Data and Value Consistency

The raw NSE datasets contain numerical fields stored as strings because of formatting characters and placeholder values.

This section investigates the numerical fields by removing thousands separators for temporary conversion, identifying values that cannot be interpreted numerically, and examining basic value ranges and logical inconsistencies.

The original datasets are preserved without modification.

In [23]:
# ============================================
# 7.4 NUMERIC DATA AND VALUE CONSISTENCY
# ============================================

# Expected numerical columns based on the NSE dataset schema
numeric_candidates = [
    "Open",
    "High",
    "Low",
    "Close",
    "LTP",
    "Settle Price",
    "Settlement Price",
    "No. of contracts",
    "Turnover",
    "Open Int",
    "Open Interest",
    "Change in OI",
    "Change in Open Interest",
    "Underlying",
    "Underlying Value",
    "Strike Price",
    "Premium Turnover"
]


def find_numeric_columns(df):
    normalized = {
        col.strip().lower(): col
        for col in df.columns
    }

    return [
        normalized[name.lower()]
        for name in numeric_candidates
        if name.lower() in normalized
    ]


def convert_numeric_column(series):
    """Convert a raw numeric column without modifying the original."""

    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)
    )

    cleaned = cleaned.replace({
        "-": np.nan,
        "": np.nan,
        "NA": np.nan,
        "N/A": np.nan
    })

    return pd.to_numeric(cleaned, errors="coerce")


def numeric_quality_analysis(df, dataset_name):

    numeric_columns = find_numeric_columns(df)
    results = []

    for column in numeric_columns:

        converted = convert_numeric_column(df[column])

        original = df[column].astype(str).str.strip()

        placeholder_count = original.eq("-").sum()
        blank_count = original.eq("").sum()

        invalid_count = (
            converted.isna().sum()
            - placeholder_count
            - blank_count
        )

        valid_values = converted.dropna()

        results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Valid Numeric Values": len(valid_values),
            "Hyphen Placeholders": placeholder_count,
            "Blank Values": blank_count,
            "Other Invalid Values": max(0, invalid_count),
            "Minimum": valid_values.min() if len(valid_values) else np.nan,
            "Maximum": valid_values.max() if len(valid_values) else np.nan,
            "Mean": valid_values.mean() if len(valid_values) else np.nan,
            "Negative Values": (valid_values < 0).sum()
        })

    return pd.DataFrame(results)


numeric_summary = pd.concat(
    [
        numeric_quality_analysis(futures_df, "Futures"),
        numeric_quality_analysis(calls_df, "Call Options (CE)"),
        numeric_quality_analysis(puts_df, "Put Options (PE)")
    ],
    ignore_index=True
)

display(numeric_summary)

,Dataset,Column,Valid Numeric Values,Hyphen Placeholders,Blank Values,Other Invalid Values,Minimum,Maximum,Mean,Negative Values
0,Futures,Open,4263,85,0,0,"13,785.00","26,770.10","20,736.66",0
1,Futures,High,4263,85,0,0,"13,896.90","26,825.00","20,833.27",0
2,Futures,Low,4263,85,0,0,"13,666.60","26,671.20","20,629.80",0
3,Futures,Close,4263,85,0,0,"13,709.10","26,772.60","20,735.38",0
4,Futures,LTP,4263,85,0,0,"13,717.25","26,795.00","20,737.00",0
5,Futures,Settle Price,4263,85,0,0,"13,709.10","26,772.60","20,735.43",0
6,Futures,No. of contracts,4263,85,0,0,260.00,"1,615,661.00","63,049.18",0
7,Futures,Open Int,4263,85,0,0,"15,300.00","21,563,945.00","4,978,355.12",0
8,Futures,Change in OI,4262,86,0,0,"-4,581,000.00","6,602,830.00","66,709.91",1087
9,Futures,Underlying Value,2891,1457,0,0,"13,634.60","26,328.55","21,869.82",0


In [24]:
# ============================================
# LOGICAL CONSISTENCY CHECKS
# ============================================

def consistency_checks(df, dataset_name):

    normalized = {
        col.strip().lower(): col
        for col in df.columns
    }

    print(f"\n{dataset_name}")
    print("=" * 55)

    # Find High and Low columns
    high_col = normalized.get("high")
    low_col = normalized.get("low")

    if high_col and low_col:

        high = convert_numeric_column(df[high_col])
        low = convert_numeric_column(df[low_col])

        comparable = high.notna() & low.notna()

        high_below_low = (
            comparable & (high < low)
        ).sum()

        print(
            f"Records where High < Low: {high_below_low:,}"
        )

    # Check negative values in price-related columns
    price_columns = [
        "Open", "High", "Low", "Close",
        "LTP", "Settle Price", "Settlement Price"
    ]

    for candidate in price_columns:

        column = normalized.get(candidate.lower())

        if column:

            values = convert_numeric_column(df[column])

            negative_count = (values < 0).sum()

            print(
                f"Negative values in {column}: {negative_count:,}"
            )


consistency_checks(futures_df, "Futures")
consistency_checks(calls_df, "Call Options (CE)")
consistency_checks(puts_df, "Put Options (PE)")


Futures
Records where High < Low: 0
Negative values in Open: 0
Negative values in High: 0
Negative values in Low: 0
Negative values in Close: 0
Negative values in LTP: 0
Negative values in Settle Price: 0

Call Options (CE)
Records where High < Low: 0
Negative values in Open: 0
Negative values in High: 0
Negative values in Low: 0
Negative values in Close: 0
Negative values in LTP: 0
Negative values in Settle Price: 0

Put Options (PE)
Records where High < Low: 0
Negative values in Open: 0
Negative values in High: 0
Negative values in Low: 0
Negative values in Close: 0
Negative values in LTP: 0
Negative values in Settle Price: 0


### 7.5 Consolidated Data Quality Summary

This section consolidates the findings from the data quality assessment.

The analysis covers missing and placeholder values, duplicate records, date validity, numerical data quality, and logical consistency. These observations will provide the basis for developing an appropriate preprocessing strategy in the next stage.

In [25]:
# ============================================
# 7.5 CONSOLIDATED DATA QUALITY SUMMARY
# ============================================

dataset_summary = []

for name, df in [
    ("Futures", futures_df),
    ("Call Options (CE)", calls_df),
    ("Put Options (PE)", puts_df)
]:

    dataset_summary.append({
        "Dataset": name,
        "Total Records": len(df),
        "Total Columns": len(df.columns),
        "Exact Duplicate Rows": df.drop(
            columns=["source_file"]
        ).duplicated().sum(),
        "Total Hyphen Placeholders": (
            df.astype(str).apply(
                lambda col: col.str.strip().eq("-")
            ).sum().sum()
        ),
        "Memory Usage (MB)": (
            df.memory_usage(deep=True).sum() / (1024 ** 2)
        )
    })


quality_overview = pd.DataFrame(dataset_summary)

display(
    quality_overview.style.format({
        "Total Records": "{:,.0f}",
        "Total Columns": "{:,.0f}",
        "Exact Duplicate Rows": "{:,.0f}",
        "Total Hyphen Placeholders": "{:,.0f}",
        "Memory Usage (MB)": "{:.2f}"
    })
)

,Dataset,Total Records,Total Columns,Exact Duplicate Rows,Total Hyphen Placeholders,Memory Usage (MB)
0,Futures,"4,348",15,54,"2,308",1.21
1,Call Options (CE),"1,319,036",18,"57,226","7,077,764",365.88
2,Put Options (PE),"1,358,362",18,"57,673","7,163,587",376.46


## 8. Conclusion

This notebook established an initial understanding of the NSE NIFTY 50 Futures and Options datasets covering January 2021 to September 2026.

The three datasets were loaded from their original quarterly CSV files while preserving the raw values and maintaining source-file information.

The exploratory data-understanding stage examined:

- Dataset structure, dimensions, and column names.
- Consistency of schemas across quarterly files.
- Record distribution across source files.
- Memory requirements.
- Missing values and placeholder entries.
- Exact duplicate records and repeated contract identifiers.
- Trading-date and expiry-date validity.
- Numerical data types, value ranges, and logical consistency.

The assessment identified differences in data availability across Futures, Call Options, and Put Options. In particular, several options-market variables contain substantial placeholder values. These observations highlight the importance of applying variable-specific preprocessing rather than indiscriminately replacing missing values or deleting observations.

No original raw dataset was modified during this notebook.

The findings from this investigation will guide the next stage of the project: data cleaning, preprocessing, and preparation of analysis-ready datasets.